# Fraud Detection & Reg E Compliance with Briefcase AI - Interactive Walkthrough

## Overview
This notebook demonstrates how to use Briefcase AI for **fraud detection with Reg E compliance**, including dispute resolution audit trails.

### What You'll Learn:
- Real-time fraud scoring with audit trail capture
- Reg E dispute timeline management
- Customer dispute linkage to original decisions
- Investigation deadline compliance tracking

### Regulatory Context:
- **Regulation**: Reg E / EFTA (Electronic Fund Transfer Act)
- **Regulator**: CFPB (Consumer Financial Protection Bureau)
- **Requirements**: Error resolution procedures, investigation timelines, dispute tracking

## Step 1: Setup and Imports

In [ ]:
import sys
import os
import uuid
import random
import hashlib
from datetime import datetime, timedelta
from typing import Dict, Any

# Add shared module to path
_p = os.path.abspath('')
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, 'shared')):
    _p = os.path.dirname(_p)
if os.path.isdir(os.path.join(_p, 'shared')):
    sys.path.insert(0, os.path.join(_p, 'shared'))

try:
    import backend
    from backend import briefcase, DecisionSnapshot, Input, Output, SqliteBackend
    print("[SUCCESS] Successfully imported Briefcase AI SDK")
except ImportError as e:
    print(f"[FAILED] Error importing required modules: {e}")

## Step 2: Initialize Briefcase AI

In [ ]:
# Initialize Briefcase AI SDK
try:
    briefcase.init_with_config(2)
    print("[SUCCESS] Briefcase AI SDK initialized")
except Exception as e:
    print(f"[FAILED] Failed to initialize SDK: {e}")

# Get configured backend
db_backend = backend.get_backend()
print("[SUCCESS] SQLite backend configured for fraud audit trails")

## Step 3: Simulate Card Transaction Data

Create a potentially fraudulent card transaction for screening.

In [ ]:
# Generate a card transaction for fraud screening
transaction_id = str(uuid.uuid4())
card_number_masked = "**** **** **** 1234"
card_number_hash = hashlib.sha256("4532123456789012".encode()).hexdigest()[:16]

transaction_data = {
    "transaction_id": transaction_id,
    "card_number_hash": card_number_hash,
    "merchant_category_code": 5967,  # Fashion/clothing
    "transaction_amount": 1250.0,   # Moderately high amount
    "transaction_timestamp": datetime.utcnow().isoformat(),
    "behavioral_risk_features_version": "v3.4.1",
    "velocity_check_window": "24h",
    "device_fingerprint_score": 0.45  # Moderate device risk
}

print("**Payment:** Card Transaction for Fraud Screening:")
for key, value in transaction_data.items():
    if key == "card_number_hash":
        print(f"  • {key}: {str(value)[:16]}...")
    else:
        print(f"  • {key}: {value}")

print(f"\n[ALERT] Risk Indicators:")
print(f"  • Amount: ${transaction_data['transaction_amount']:,.2f} (above average)")
print(f"  • Device risk score: {transaction_data['device_fingerprint_score']} (moderate)")
print(f"  • Merchant category: Fashion (higher fraud risk)")

## Step 4: Real-Time Fraud Detection Model

Simulate the AI model that performs real-time fraud scoring.

In [ ]:
def simulate_fraud_detection_model(transaction_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simulates a real-time fraud detection AI model.
    In production, this would use sophisticated ML algorithms
    with behavioral analytics, device fingerprinting, etc.
    """
    amount = transaction_data["transaction_amount"]
    mcc = transaction_data["merchant_category_code"]
    device_score = transaction_data["device_fingerprint_score"]

    fraud_score = 0.0

    # Amount-based risk
    if amount > 1000:
        fraud_score += 0.3
    elif amount > 500:
        fraud_score += 0.15

    # MCC-based risk (fashion/jewelry = higher risk)
    high_risk_mccs = [5944, 5945, 5967, 5970]  # Fashion, jewelry
    if mcc in high_risk_mccs:
        fraud_score += 0.25

    # Device fingerprint risk
    if device_score > 0.6:
        fraud_score += 0.4
    elif device_score > 0.3:
        fraud_score += 0.2

    # Add some randomness to simulate model uncertainty
    fraud_score += random.uniform(-0.1, 0.2)
    fraud_score = max(0.0, min(1.0, fraud_score))  # Clamp between 0-1

    # Decision thresholds
    decision_threshold = 0.7
    
    if fraud_score >= decision_threshold:
        decision = "block"
        risk_level = "high"
    elif fraud_score >= 0.4:
        decision = "step_up_auth"  # Additional authentication required
        risk_level = "medium"
    else:
        decision = "approve"
        risk_level = "low"

    return {
        "decision": decision,
        "fraud_score": round(fraud_score, 3),
        "risk_level": risk_level,
        "decision_threshold": decision_threshold,
        "model_version": "fraud-detection-v5.2.1",
        "processing_time_ms": random.randint(45, 95)
    }

# Run fraud detection
print("[AUTOMATED] Running real-time fraud detection...")
fraud_result = simulate_fraud_detection_model(transaction_data)

print(f"\n**Results:** Fraud Detection Results:")
print(f"  • Decision: {fraud_result['decision'].upper()}")
print(f"  • Fraud Score: {fraud_result['fraud_score']}")
print(f"  • Decision Threshold: {fraud_result['decision_threshold']}")
print(f"  • Risk Level: {fraud_result['risk_level']}")
print(f"  • Processing Time: {fraud_result['processing_time_ms']}ms")

if fraud_result['decision'] == 'block':
    print(f"\n🚫 TRANSACTION BLOCKED - Customer will see: 'Transaction declined'")
    print(f"**Insight:** Customer may dispute this decision under Reg E")

## Step 5: Create Fraud Decision Audit Trail

Capture the fraud decision with Reg E compliance metadata.

In [ ]:
# Prepare Reg E regulatory metadata
regulatory_metadata = {
    "regulation": "Reg E",
    "electronic_fund_transfer": True,
    "customer_dispute_eligible": True,
    "error_resolution_applicable": fraud_result["decision"] == "block",
    "decision_timestamp": datetime.utcnow().isoformat(),
    "model_explainability_available": True,
    "investigation_required_if_disputed": True
}

print("**Details:** Reg E Compliance Metadata:")
for key, value in regulatory_metadata.items():
    print(f"  • {key}: {value}")

# Create decision snapshot for fraud detection
try:
    decision_snapshot = backend.create_decision_snapshot(
        function_name="fraud_detection_decision",
        inputs=transaction_data,
        outputs=fraud_result,
        metadata=regulatory_metadata,
        input_types={
            "transaction_amount": "float",
            "merchant_category_code": "int",
            "device_fingerprint_score": "float"
        },
        output_types={
            "fraud_score": "float",
            "decision_threshold": "float"
        }
    )
    print("\n[SUCCESS] Fraud decision snapshot created")
    
except Exception as e:
    print(f"\n[FAILED] Error creating decision snapshot: {e}")

## Step 6: Store in Immutable Audit Trail

In [ ]:
# Store fraud decision in backend
try:
    stored_decision_id = db_backend.save_decision(decision_snapshot)
    print(f"[SUCCESS] Fraud decision stored in audit trail")
    print(f"[SECURED] Decision ID: {stored_decision_id}")
    
    if fraud_result["decision"] == "block":
        print(f"\n🚫 TRANSACTION BLOCKED - Reg E Implications:")
        print(f"  • Customer can dispute within 60 days")
        print(f"  • Bank has 10 business days to investigate")
        print(f"  • Must provide provisional credit if investigation takes longer")
    
except Exception as e:
    print(f"[FAILED] Error storing decision: {e}")

## Step 7: Simulate Customer Dispute (If Transaction Was Blocked)

Show how customer disputes are linked to original fraud decisions.

In [ ]:
dispute_id = None
updated_decision_id = stored_decision_id

if fraud_result["decision"] == "block":
    print("📞 SIMULATING CUSTOMER DISPUTE")
    print("=" * 50)
    print("Customer calls: 'My card was declined at XYZ Store for $1,250'")
    print("Customer claims: 'This was a legitimate purchase'")
    
    # Generate dispute ID
    dispute_id = str(uuid.uuid4())
    print(f"\n🎫 Dispute ID generated: {dispute_id}")
    
    # Create new decision snapshot for dispute linkage
    try:
        original_decision = db_backend.load_decision(stored_decision_id)
        if original_decision:
            # Create a new decision snapshot for the dispute linkage event
            dispute_metadata = {
                "linked_original_decision": stored_decision_id,
                "dispute_id": dispute_id,
                "dispute_filed_timestamp": datetime.utcnow().isoformat(),
                "investigation_deadline": (datetime.utcnow() + timedelta(days=10)).isoformat(),
                "regulation": "Reg E",
                "investigation_required": True,
                "reg_e_timeline_compliant": True
            }

            # Create dispute linkage decision
            dispute_decision = backend.create_decision_snapshot(
                function_name="fraud_dispute_filing",
                inputs={"original_decision_id": stored_decision_id, "dispute_id": dispute_id},
                outputs={"dispute_status": "filed", "investigation_status": "initiated"},
                metadata=dispute_metadata
            )

            # Store the dispute decision
            updated_decision_id = db_backend.save_decision(dispute_decision)
            print(f"[SUCCESS] Dispute decision created and linked: {updated_decision_id}")

    except Exception as e:
        print(f"[FAILED] Error creating dispute linkage: {e}")

else:
    print("ℹ Transaction was not blocked, so no dispute scenario to simulate")

## Step 8: Retrieve and Display Complete Audit Trail

In [ ]:
print("**Analysis:** COMPLETE AUDIT TRAIL DEMONSTRATION")
print("=" * 65)

# Load the most recent decision (dispute linkage if applicable, otherwise original)
try:
    retrieved_decision = db_backend.load_decision(updated_decision_id)
    if retrieved_decision:
        backend.print_audit_summary(retrieved_decision)
    else:
        print("[FAILED] Failed to retrieve decision from backend")
        
except Exception as e:
    print(f"[FAILED] Error retrieving decision: {e}")

## Step 9: CFPB Investigator Simulation

In [ ]:
print("🏛 CFPB INVESTIGATOR SIMULATION")
print("=" * 65)

if dispute_id:
    investigator_query = f"Show me all records related to customer dispute {dispute_id} including the original fraud decision"
else:
    investigator_query = f"Show me the fraud detection decision for transaction {transaction_id}"

print(f"**Analysis:** INVESTIGATOR QUERY: {investigator_query}")

investigator_response = backend.format_examiner_response(
    stored_decision_id,  # Always show the original fraud decision
    investigator_query,
    db_backend
)
print(investigator_response)

## Step 10: Reg E Timeline Compliance Check

In [ ]:
print("⏰ REG E COMPLIANCE VALIDATION")
print("=" * 65)

if dispute_id:
    # Check investigation timeline for disputed transaction
    if retrieved_decision.tags.get("investigation_deadline"):
        deadline_str = retrieved_decision.tags["investigation_deadline"]
        deadline = datetime.fromisoformat(deadline_str.replace('Z', '+00:00') if deadline_str.endswith('Z') else deadline_str)
        days_remaining = (deadline - datetime.utcnow()).days

        print(f"📅 Investigation deadline: {deadline.strftime('%Y-%m-%d')}")
        print(f"⏰ Days remaining: {days_remaining}")

        if days_remaining >= 0:
            print("[SUCCESS] Investigation timeline compliant")
        else:
            print("[FAILED] Investigation deadline exceeded - SLA violation")

    # Validate decision reconstructability
    original_decision = db_backend.load_decision(stored_decision_id)
    model_version = None
    for output in original_decision.outputs:
        if output.name == 'model_version':
            model_version = output.value
            break
            
    if model_version:
        print(f"\n[SUCCESS] Model version preserved: {model_version}")
        print("[SUCCESS] Decision fully reconstructable for dispute resolution")
    else:
        print("[FAILED] Model version missing - may impact dispute resolution")
else:
    print("ℹ No dispute filed - standard fraud detection compliance applies")
    print("[SUCCESS] Decision properly captured with Reg E metadata")

# Overall Reg E compliance validation
required_fields = [
    "regulation",
    "electronic_fund_transfer",
    "customer_dispute_eligible",
    "decision_timestamp"
]

# Use original decision for compliance validation
original_decision = db_backend.load_decision(stored_decision_id)
validation_result = backend.validate_regulatory_completeness(
    original_decision,
    required_fields
)

status_icon = "[SUCCESS]" if validation_result['is_compliant'] else "[FAILED]"
status_text = "COMPLIANT" if validation_result['is_compliant'] else "NON-COMPLIANT"

print(f"\n{status_icon} Reg E Compliance Status: {status_text}")
print(f"**Results:** Completeness Score: {validation_result['completeness_score']:.1%}")

## Step 11: Complete Audit Chain Summary

In [ ]:
print("🔗 COMPLETE AUDIT CHAIN")
print("=" * 40)

print(f"1. **Payment:** Transaction processed: {transaction_data['transaction_timestamp']}")
print(f"2. [AUTOMATED] Fraud decision made: {fraud_result['decision']} (score: {fraud_result['fraud_score']})")
print(f"3. [SECURED] Decision stored: {stored_decision_id}")
if dispute_id:
    retrieved_deadline = retrieved_decision.tags.get('investigation_deadline', 'N/A')
    print(f"4. 📞 Customer dispute filed: {dispute_id}")
    print(f"5. ⏰ Investigation deadline: {retrieved_deadline[:10] if retrieved_deadline != 'N/A' else 'N/A'}")
    print(f"6. 🔗 Dispute linked to original decision: {updated_decision_id}")
print(f"{'7' if dispute_id else '4'}. **Details:** Full audit trail retrievable on demand")

print(f"\n[SUCCESS] Fraud detection & Reg E audit trail demonstration completed")
print(f"🆔 Original Decision ID: {stored_decision_id}")
if dispute_id:
    print(f"🎫 Dispute ID: {dispute_id}")
    print(f"🔗 Linked Decision ID: {updated_decision_id}")

## Summary

### What We Accomplished
[SUCCESS] **Created a complete Reg E compliant fraud detection audit trail**

[SUCCESS] **Captured all critical elements:**
- Real-time fraud scoring and decision logic
- Complete transaction context and risk factors
- Customer dispute linkage (when applicable)
- Investigation timeline tracking

[SUCCESS] **Demonstrated Reg E compliance:**
- Dispute eligibility tracking
- Investigation deadline management
- Decision reconstructability for disputes
- Complete audit chain from transaction to resolution

### Key Reg E Compliance Benefits
- **Error Resolution**: Complete decision history for dispute investigation
- **Timeline Tracking**: Automated deadline management for investigations
- **Decision Linkage**: Connect disputes back to original fraud decisions
- **Audit Transparency**: Full visibility into fraud detection reasoning

### Critical Reg E Requirements Met
**Details:** **Electronic Fund Transfer Documentation:**
- Transaction details and timing
- Fraud detection methodology
- Decision rationale and confidence
- Customer communication records (implied)

⏰ **Investigation Timeline Management:**
- 10 business day investigation deadline
- Provisional credit requirements
- Final determination documentation
- Customer notification requirements

🔗 **Dispute Resolution Audit Trail:**
- Original decision preservation
- Dispute filing timestamp
- Investigation progress tracking
- Resolution documentation

### Next Steps in Production
1. **Integration**: Connect to actual fraud detection models
2. **Automation**: Automated dispute intake and timeline tracking
3. **Notifications**: Customer communication automation
4. **Reporting**: Regular Reg E compliance reporting

**Transaction**: `{transaction_id}`  
**Fraud Decision**: `{stored_decision_id}`  
**Dispute**: `{dispute_id if dispute_id else 'N/A'}`